# Iceberg → Parquet on S3: a silver landing zone for Databricks

Exports Iceberg tables of the Cloudera lakehouse as plain Parquet into the project's folder on the
environment bucket. Databricks can read that folder as a silver layer and build Unity Catalog Delta tables
from it.

The notebook imports the same module as the CDE job `rsingh-spend-anl-export-parquet`
(`cde/jobs/export_parquet.py`), so both produce the same output.

| | |
|---|---|
| Source | `rsingh_spend_anl_silver.spend_event` (Iceberg v2, ~30M rows); any `layer.table` works |
| Target | `s3a://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export/<layer>/<table>/` |
| Format | Parquet, snappy, `TIMESTAMP_MICROS` (UTC), Hive-style `<col>_month=YYYY-MM` folders |
| Proof | the snapshot ID it read, a row-count check, and `_export_manifest.json` written last |

**Why through Spark and not a file copy:** an Iceberg table's data files aren't the table. Gold's MERGE
loads leave position-delete files behind, and a plain Parquet reader such as Databricks ignores them, so
it would count every reloaded row twice. Reading the snapshot through Spark applies the deletes.

**Session:** New Session › JupyterLab, Python 3.11 › **Enable Spark: Spark 3.5.4 - CDE 1.25.2** ›
2 vCPU / 8 GB. The executors run on the workspace's Kubernetes cluster.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "cde" / "jobs"))
import export_parquet as ex

DB_PREFIX = "rsingh_spend_anl"
TABLES = ["silver.spend_event"]          # add e.g. "silver.customer", "gold.spend_features"
TARGET = ex.DEFAULT_TARGET               # s3a://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export
BUCKET = "/".join(TARGET.split("/")[:3])
print("target:", TARGET)

## 1. Spark with Iceberg

The Spark add-on ships the Iceberg runtime in `/opt/spark/optional-lib`. `SparkSessionCatalog` puts
Iceberg in front of the Hive metastore, so the tables resolve by the same names the CDE jobs use.
`spark.kerberos.access.hadoopFileSystems` gets the session an IDBroker token for the bucket, for both
reading and writing.

In [ ]:
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

jars = ",".join(sorted(glob.glob("/opt/spark/optional-lib/iceberg-*.jar")))
builder = (SparkSession.builder.appName("spend-export-parquet-notebook")
           .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
           .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
           .config("spark.sql.catalog.spark_catalog.type", "hive")
           .config("spark.hadoop.iceberg.engine.hive.enabled", "true")
           .config("spark.kerberos.access.hadoopFileSystems", BUCKET)
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.executor.instances", "2").config("spark.executor.cores", "2")
           .config("spark.executor.memory", "6g"))
if jars:
    builder = builder.config("spark.jars", jars)
spark = builder.getOrCreate()
print("Spark", spark.version, "| Iceberg jars:", jars or "already on the classpath")

## 2. The source: what Iceberg holds

Current snapshot, its recent history, and the data vs delete files it is made of. Run this cell with
`gold.spend_features` to see the delete files that a raw copy would get wrong.

In [ ]:
for spec in TABLES:
    layer, table = spec.split(".")
    fq = f"{DB_PREFIX}_{layer}.{table}"
    sid, at = ex.current_snapshot(spark, fq)
    print(f"\n== {fq}: current snapshot {sid} ({at})")
    spark.sql(f"SELECT made_current_at, snapshot_id, is_current_ancestor FROM {fq}.history "
              "ORDER BY made_current_at DESC LIMIT 5").show(truncate=False)
    (spark.sql(f"SELECT CASE content WHEN 0 THEN 'data' WHEN 1 THEN 'position deletes' ELSE 'equality deletes' END "
               f"AS file_type, COUNT(*) AS files, SUM(record_count) AS records, "
               f"ROUND(SUM(file_size_in_bytes) / 1e6, 1) AS mb FROM {fq}.files GROUP BY content ORDER BY content")
     .show(truncate=False))

## 3. Dry run: resolve snapshots, count rows, write nothing

In [ ]:
plan = ex.run(spark, DB_PREFIX, ex.parse_tables(",".join(TABLES)), TARGET, dry_run=True)
for m in plan:
    print(f"{m['source_table']} @ {m['iceberg_snapshot_id']}: {m['row_count']:,} rows, "
          f"{m['source_delete_files']} delete files -> {m['target_path']} (partition {m['partition_column']})")

## 4. Export

Each table's folder is replaced. Spark writes `_SUCCESS`; the job reads the Parquet back, fails if the row
count differs from the snapshot, then writes `_export_manifest.json` last.

In [ ]:
manifests = ex.run(spark, DB_PREFIX, ex.parse_tables(",".join(TABLES)), TARGET)
for m in manifests:
    print(f"{m['source_table']} @ {m['iceberg_snapshot_id']}: {m['row_count']:,} rows in {m['files']} files, "
          f"{m['bytes'] / 1e6:,.1f} MB -> {m['target_path']}")

## 5. Check it the way Databricks will see it

Read the folder as plain Parquet (no Iceberg) and compare it month by month with the Iceberg snapshot.
Then list the S3 objects.

In [ ]:
m = manifests[0]
pq = spark.read.parquet(m["target_path"])
pq.printSchema()

part, src_col = m["partition_column"], m["partition_source"]
if part:
    ice = (spark.sql(f"SELECT * FROM {m['source_table']} VERSION AS OF {m['iceberg_snapshot_id']}")
           .groupBy(F.date_format(src_col, "yyyy-MM").alias(part)).count().withColumnRenamed("count", "iceberg"))
    cmp = pq.groupBy(part).count().withColumnRenamed("count", "parquet").join(ice, part, "full")
    cmp.withColumn("match", F.col("parquet") == F.col("iceberg")).orderBy(part).show(40, truncate=False)
pq.limit(5).toPandas()

In [ ]:
jvm = spark._jvm
root = jvm.org.apache.hadoop.fs.Path(m["target_path"])
fs = root.getFileSystem(spark._jsc.hadoopConfiguration())
base = str(fs.makeQualified(root)) + "/"
it = fs.listFiles(root, True)
rows = []
while it.hasNext():
    s = it.next()
    rows.append((str(s.getPath()).replace(base, ""), round(s.getLen() / 1e6, 2)))
print(f"{len(rows)} objects under {m['target_path']}")
for name, mb in sorted(rows)[:12]:
    print(f"  {mb:>8} MB  {name}")
print(json.dumps({k: m[k] for k in ("iceberg_snapshot_id", "row_count", "files", "exported_at")}, indent=1))

## 6. On the Databricks side: Unity Catalog Delta tables

One-time setup by the Databricks admin: a **storage credential** (an IAM role that can read
`s3://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export/`, trusted by the Databricks account)
and an **external location** on that prefix. The bucket owner adds that role to the bucket policy.
Databricks uses `s3://`, not `s3a://`.

```sql
CREATE EXTERNAL LOCATION IF NOT EXISTS spend_cloudera_export
  URL 's3://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export/'
  WITH (STORAGE CREDENTIAL <credential>);
CREATE SCHEMA IF NOT EXISTS <catalog>.spend_silver;
```

Then, per table, the statement printed below: a **managed** Delta table, refreshed by running it again
after each export. Copying keeps the export folder a plain landing zone. `CONVERT TO DELTA` in place
would make the next export overwrite a Delta table's files. `txn_date_month` is only the folder layout,
so it is dropped; liquid clustering on `txn_date` replaces it. Incremental alternatives: `COPY INTO`
or Auto Loader on the same path, triggered when `_export_manifest.json` shows a newer `exported_at`.

In [ ]:
for m in manifests:
    print(f"-- {m['source_table']} @ snapshot {m['iceberg_snapshot_id']}, {m['row_count']:,} rows\n{m['databricks_sql']}\n")

## The same export as a CDE job

```bash
cde job run --name rsingh-spend-anl-export-parquet --wait     # default: silver.spend_event, customer, account
cde job run --name rsingh-spend-anl-export-parquet --arg=--db-prefix --arg=rsingh_spend_anl \
    --arg=--tables --arg=gold.spend_features --wait
```

It isn't in the nightly DAG. To refresh Databricks every night, add it as a task after `dq_gold`
(silver and gold are final by then).

In [ ]:
spark.stop()